This will be an implementation of https://pvitelli.net/2020/01/20/origami-flagstone-tessellations/

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%pylab inline
import eucare as ec
from eucare.overlap import CREASE_ASSIGNMENT, MOUNTAIN, VALLEY

In [ ]:
def make_flagstone(G, t=1/3, symmetric_pleats=True, assign_creases=True, **reciprocal_figure_kwargs):
    # set up the topology of the CP
    f = next(iter(G.faces))
    if 'reciprocal_pos' not in f or len(reciprocal_figure_kwargs):
        print('Calculating reciprocal figure..')
        ec.reciprocal_figures.reciprocal_figure(G, **reciprocal_figure_kwargs)
        print('Done with reciprocal figure.')
    for f in G.faces:
        f['midpoint'] = f['reciprocal_pos']
    
#     ec.conway.expand_graph()(G).show()
    CP = ec.conway.flagstone_pvitelli_graph(t)(G, delete_on_border=False, copy_graph=True)
    
    v_dict = {}
    # correct the geometry of the CP
    to_process = []

    vNs = [v for v in CP.vertices if v.get('label', None) == 'N1']
    vMs = [v for v in CP.vertices if v.get('label', None) == 'N2']
    for v_orig in G.vertices:
        if v_orig.on_border():
            continue
        for h_orig in v_orig.outgoing_iter():
            vM = next(v for v in vMs if v.get('pre_conway', None) == h_orig)
            vN = next(v for v in vNs if v.get('pre_conway', None) == h_orig.rev)
            h = next(h for h in vM.outgoing_iter() if h.dest is vN).rev
            vL = h.nex.dest
            vV1 = h.nex.rev.nex.dest
            vV2 = h.pre.rev.nex.dest
            
            pos = np.stack([v['pos'] for v in (vV1, vV2, vN, vM, vL, v_orig)])
            v_dict[(h_orig.face, h_orig.orig)] = vV1
            to_process.append({
                'P': v_orig,
                'h_orig': h_orig,
                'V1': vV1,
                'V2': vV2,
                'L': vL,
                'M': vM,
                'N': vN,
            })
    
#     v_dict = {}
#     for f in CP.faces:
#         if isinstance(f.get('pre_conway', None), ec.half.Vertex):
#             for v in f.vertex_iter():
#                 print(v.get('pre_conway'))
#                 if isinstance(v.get('pre_conway', None), ec.half.HalfEdge):
#                     # get original face and assign to v_dict
#                     f_orig = v['pre_conway'].face
#                     if f_orig is not None:
#                         v_dict[(f_orig, f['pre_conway'])] = v
#                         print('as')
            
    # first, assign correct positions to v1 and v2
    # start with some initial original face and proceed in a breathd-first manner
    f_initial = next(iter(G.faces))
    f_initial['reciprocal_offset'] = np.zeros(2)
    dummy = 0
    for v_orig in f_initial.vertex_iter():
        if (f_initial, v_orig) in v_dict:
            v_dict[(f_initial, v_orig)]['pos'] =  v_orig['pos']
    for (f1, f2) in ec.search_trees.face_bfs_tree(f_initial):
        print(f1, f2)
        f2['reciprocal_offset'] = f1['reciprocal_offset'] + t * (f2['reciprocal_pos'] - f1['reciprocal_pos'])
        for v_orig in f2.vertex_iter():
            if (f2, v_orig) in v_dict:
                v_dict[(f2, v_orig)]['pos'] = f2['reciprocal_offset'] + v_orig['pos']
                print('..')
            else:
                print('asdf')
    CP.show()

    for data in to_process:
        v_orig, h_orig, vV1, vV2, vL, vM, vN = data.values()
        #FIXME choose pP in a more reasonable way
        pP, pV1, pV2 = v_orig['pos'], vV1['pos'], vV2['pos']
        l = np.linalg.norm(pV1 - pV2)
        if symmetric_pleats:
            pE = (pV1 + pV2) / 2
        else:
            pE = ec.base.line_intersection([pV1, pV2], [h_orig.orig['pos'], h_orig.dest['pos']])
        
        c = np.linalg.norm(pV1 - pE)

        rot_mat = ec.base.rotation_matrix(ec.base.angle_to_axis(pV2-pV1)) 
        inv_rot_mat = ec.base.rotation_matrix(-ec.base.angle_to_axis(pV2-pV1)) 

        pPx, pPy = rot_mat @ (pP - pV1)

        pL = np.array([l/2, (pPx**2 - pPx * l + pPy**2) / (2 * pPy)])
        pM = np.array([c/2, (pPx**2 - pPx * l) / (2 * pPy) + (l - pPx)/2 * (pPx-c)/pPy])
        pN = np.array([(c+l)/2, (pPx**2 - pPx * l) / (2 * pPy) - pPx/2 * (pPx-c)/pPy])

        pL, pM, pN = (inv_rot_mat @ np.stack([pL, pM, pN], axis=1)).T + pV1[None]

        vL['pos'] = pL
        vM['pos'] = pM
        vN['pos'] = pN
    
    CP.show()
    
    # cut off the borders
    G.simply_connected = True
    fs_mult = [h.rev.face for h in G.border_edge_iter()]
    fs = []
    for f in fs_mult:
        fnew = next(fnew for fnew in CP.faces if fnew.get('pre_conway', None) is f)
        if fnew not in fs:
            fs.append(fnew)
    poly = np.stack([f.midpoint() for f in fs])
    ec.cutting.cut_out_poly(CP, poly)
    CP.recompute_lengths_and_angles()
    for v in list(CP.vertices):
        if v.on_border() and v.order()==2 and np.allclose(v.angle_sum(), np.pi):
            CP.join_vertex(v)
    
    # adjust positions of points on border
    if symmetric_pleats:
        fs = (f for f in CP.faces if f.on_border() 
              and isinstance(f.get('pre_conway', None), ec.half.Face))
        hs = (next(h.rev.pre.rev for h in f.halfedge_iter() if h.nex.rev.on_border() and not h.rev.on_border())
             for f in fs)
        for h in hs:
            v1, v2, v3, v4 = h.orig, h.dest, h.nex.nex.orig, h.nex.nex.dest
            v2['pos'] = 3/4 * v1['pos'] + 1/4 * v4['pos']
            v3['pos'] = 1/4 * v1['pos'] + 3/4 * v4['pos']
    
    CP.recompute_lengths_and_angles()
    
    if assign_creases:
        for f in CP.faces:        
            if isinstance(f.get('pre_conway', None), ec.half.Vertex):
                for h in f.halfedge_iter():
                    for h2 in [
                        h.rev.nex, h.rev.pre,
                        h.rev.nex.rev.nex.rev.pre, h.rev.pre.rev.pre.rev.nex,
                        h.rev.nex.rev.pre.rev.pre, h.rev.pre.rev.nex.rev.nex
                    ]:
                        h2[CREASE_ASSIGNMENT] = h2.rev[CREASE_ASSIGNMENT] = MOUNTAIN
        for h in CP.halfedges:
            if not h.on_border() and not CREASE_ASSIGNMENT in h:
                h[CREASE_ASSIGNMENT] = h.rev[CREASE_ASSIGNMENT] = VALLEY
        ec.overlap.color_creases(CP)
    return CP

# G = ec.example_graphs.from_tiles(ec.example_tilesets.curved_omnitruncate(6, 3), rings=2)
G = ec.example_graphs.from_tiles(ec.example_tilesets.curved_platonic(4, 4), rings=1)
ps, vs = G.get_position_view()
ps[:, 0] *=1.3
# G = ec.conway.dual_graph()(G)
G.convert_to_euclidean()
G.recompute_lengths_and_angles()
G.show()
# for f in G.faces:
#     f['reciprocal_pos'] = f.pseudo_incenter()

# CP = make_flagstone(G, 0.42)
CP = make_flagstone(G, 0.1, symmetric_pleats=True)
CP.show(render_faces=True, render_vertices=False, height=4000)

vs = [v for v in CP.vertices if not np.any([v2.on_border() for v2 in v.vertex_iter()])]
print(len(vs))
print(ec.reciprocal_figures.max_kawasaki_sum(vs) * 180 / np.pi)
print(ec.reciprocal_figures.max_kawasaki_sum(CP) * 180 / np.pi)

In [ ]:
print(ec.reciprocal_figures.max_kawasaki_sum(CP))
from eucare.overlap import fold_complete

result = fold_complete(CP.copy(), overlap_eps=1e-6, area_eps=0)

In [ ]:
import os
from eucare.redering import SvgwriteRenderer
from eucare.overlap import save_results

path = 'nice_images/pvitelli_flagstones/6.4.12_real'
bbox = (25, 20)
# bbox = (5, 20)
# bbox = (35, 30)
# bbox = (65, 48)
# bbox = (95, 58)

render_settings = dict(face_inset=0, render_vertices=False, render_faces=False, height=2048)
save_results(result, path, bbox=bbox, render_settings=render_settings)